# Cross-Signer Generalization on WLASL — RQE + Transformer

Evaluation notebook for the CV2 final project. We test whether a Transformer encoder fed with **Relative Quantization Encoding (RQE)** of 3D skeletal landmarks generalizes across unseen signers.

*Note:* this notebook is evaluation-only. No training loop here — we assume a checkpoint exists (or we just run a forward pass for sanity).

## 1. Setup & Imports

In [ ]:
import os
import glob
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score

# Kaggle-friendly device setup (T4 GPU when available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Repro
torch.manual_seed(0)
np.random.seed(0)

## 2. Dataset & Preprocessing (RQE)

Each sample is a `(T, J, 3)` numpy array of MediaPipe-style 3D landmarks. Landmark indexing follows MediaPipe Holistic / Pose convention: indices 11 and 12 are the left and right shoulder.

RQE steps applied per-frame:
1. Root = midpoint of shoulders (joints 11 & 12).
2. Subtract root from every joint → translational invariance.
3. Torso length = ||shoulder_L - shoulder_R||; divide everything by it → scale/subject invariance.

Temporal length is fixed to 64 frames (pad with zeros or center-truncate).

In [ ]:
class WLASLDataset(Dataset):
    """WLASL dataset over pre-extracted 3D skeletal landmarks.

    Expects a list of (npy_path, label) tuples. Each .npy is shaped (T, J, 3).
    Applies RQE normalization and pads/truncates to `max_frames`.
    """

    LEFT_SHOULDER = 11
    RIGHT_SHOULDER = 12
    EPS = 1e-6

    def __init__(self, samples, max_frames=64):
        # samples: list of dicts or tuples like {'path': ..., 'label': int, 'signer_id': str}
        self.samples = samples
        self.max_frames = max_frames

    def __len__(self):
        return len(self.samples)

    def _rqe(self, kp):
        # kp: (T, J, 3)
        ls = kp[:, self.LEFT_SHOULDER, :]   # (T, 3)
        rs = kp[:, self.RIGHT_SHOULDER, :]  # (T, 3)
        root = 0.5 * (ls + rs)              # shoulder midpoint, (T, 3)

        # 1) translation invariance — subtract root from all joints
        kp = kp - root[:, None, :]

        # 2) scale invariance — divide by torso length (per-frame)
        torso = np.linalg.norm(ls - rs, axis=-1, keepdims=True)  # (T, 1)
        torso = np.clip(torso, self.EPS, None)
        kp = kp / torso[:, :, None]
        return kp.astype(np.float32)

    def _temporal_fix(self, kp):
        T = kp.shape[0]
        if T == self.max_frames:
            return kp, np.ones(T, dtype=np.bool_)
        if T > self.max_frames:
            # center-crop
            start = (T - self.max_frames) // 2
            kp = kp[start:start + self.max_frames]
            return kp, np.ones(self.max_frames, dtype=np.bool_)
        # pad with zeros at the end
        pad = np.zeros((self.max_frames - T, *kp.shape[1:]), dtype=kp.dtype)
        mask = np.concatenate([np.ones(T, dtype=np.bool_),
                                np.zeros(self.max_frames - T, dtype=np.bool_)])
        kp = np.concatenate([kp, pad], axis=0)
        return kp, mask

    def __getitem__(self, idx):
        item = self.samples[idx]
        path = item['path']
        label = item['label']

        kp = np.load(path)  # (T, J, 3)
        kp = self._rqe(kp)
        kp, mask = self._temporal_fix(kp)

        # flatten joints+coords per frame -> (T, J*3)
        T, J, C = kp.shape
        kp = kp.reshape(T, J * C)

        return {
            'x': torch.from_numpy(kp),                       # (T, J*3)
            'mask': torch.from_numpy(mask),                  # (T,)
            'y': torch.tensor(label, dtype=torch.long),
            'signer_id': item.get('signer_id', 'unknown'),
        }

## 3. Model Architecture — RQETransformer

In [ ]:
class RQETransformer(nn.Module):
    """Transformer encoder over RQE-normalized landmark sequences.

    Input:  (B, T, J*3)
    Output: (B, num_classes)
    """

    def __init__(self, num_joints, num_classes,
                 d_model=256, nhead=8, num_layers=4,
                 dim_feedforward=512, dropout=0.1, max_len=64):
        super().__init__()
        in_dim = num_joints * 3

        # Linear projection from raw landmarks to d_model
        self.input_proj = nn.Linear(in_dim, d_model)

        # Learned positional embedding (simpler than sinusoidal, works fine for T<=64)
        self.pos_embed = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x, mask=None):
        # x: (B, T, J*3); mask: (B, T) — True = valid frame
        B, T, _ = x.shape
        h = self.input_proj(x) + self.pos_embed[:, :T, :]

        # PyTorch transformer wants True = ignore in src_key_padding_mask
        key_padding_mask = None if mask is None else (~mask)
        h = self.encoder(h, src_key_padding_mask=key_padding_mask)
        h = self.norm(h)

        # Global average pooling over valid frames
        if mask is not None:
            m = mask.float().unsqueeze(-1)  # (B, T, 1)
            h = (h * m).sum(dim=1) / m.sum(dim=1).clamp(min=1e-6)
        else:
            h = h.mean(dim=1)

        return self.head(h)

## 4. Evaluation Loop

Strictly held-out test set. We compute Top-1 accuracy and macro-F1 (better for the class-imbalanced WLASL setting). Predictions and labels are returned so we can later slice them by signer for the failure analysis.

In [ ]:
@torch.no_grad()
def evaluate_model(model, test_loader, criterion, device=device):
    model.eval()

    all_preds, all_labels, all_signers = [], [], []
    total_loss, n = 0.0, 0

    for batch in test_loader:
        x = batch['x'].to(device, non_blocking=True)
        y = batch['y'].to(device, non_blocking=True)
        mask = batch['mask'].to(device, non_blocking=True)

        logits = model(x, mask=mask)
        loss = criterion(logits, y)

        preds = logits.argmax(dim=-1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(y.cpu().numpy())
        all_signers.extend(batch['signer_id'])

        total_loss += loss.item() * y.size(0)
        n += y.size(0)

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    top1 = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    avg_loss = total_loss / max(n, 1)

    print(f'Test samples : {n}')
    print(f'Loss         : {avg_loss:.4f}')
    print(f'Top-1 acc.   : {top1*100:.2f}%')
    print(f'Macro F1     : {macro_f1*100:.2f}%')

    return {
        'preds': all_preds,
        'labels': all_labels,
        'signers': np.array(all_signers),
        'top1': top1,
        'macro_f1': macro_f1,
        'loss': avg_loss,
    }

### Wiring example

Fill in `test_samples` with your held-out (unseen-signer) split, point `CKPT_PATH` to the trained weights, and run.

In [ ]:
# ---- config ----
NUM_JOINTS  = 75      # depends on the landmark extractor (e.g. MediaPipe Holistic subset)
NUM_CLASSES = 100     # WLASL100 / WLASL300 / WLASL2000 — pick your subset
MAX_FRAMES  = 64
BATCH_SIZE  = 32
CKPT_PATH   = '/kaggle/input/rqe-transformer-weights/rqe_transformer.pt'  # change me

# ---- test samples (held-out signers) ----
# Each entry: {'path': '/path/to/landmarks.npy', 'label': int, 'signer_id': str}
test_samples = []  # populate from your split file

test_set    = WLASLDataset(test_samples, max_frames=MAX_FRAMES)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

model = RQETransformer(num_joints=NUM_JOINTS, num_classes=NUM_CLASSES,
                        d_model=256, nhead=8, num_layers=4,
                        dim_feedforward=512, dropout=0.1, max_len=MAX_FRAMES).to(device)

# Load pretrained weights if available
if os.path.isfile(CKPT_PATH):
    state = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(state.get('model', state), strict=False)
    print('Loaded checkpoint:', CKPT_PATH)
else:
    print('No checkpoint found at', CKPT_PATH, '— running with random init (sanity only).')

criterion = nn.CrossEntropyLoss()

# Only run if we actually have test samples loaded
if len(test_set) > 0:
    results = evaluate_model(model, test_loader, criterion, device=device)